# Step 6 – Independent validation for Koshi with Sentinel-2
1. Finds the **clearest Sentinel-2 image** over the Koshi box between 28 Sep and 15 Oct 2024.
2. Builds **independent water labels** from it (MNDWI, clouds masked) and a **dry-season water mask** (Feb–Apr 2024 median).
3. Scores every method against these labels at two levels:
   - **Level 1 – water extent on the flood date:** threshold, U-Net A, U-Net B.
   - **Level 2 – new water vs normal:** change detection, A, B, UNOSAT.

Runtime: **GPU (T4)**, about 20 min. Needs Earth Engine and Drive with the earlier outputs.
Optional: upload the UNOSAT SHP zip to `/content` (Files panel) to include UNOSAT in level 2.

In [ ]:
PROJECT = 'your-ee-project-id'   # <-- your Earth Engine Cloud project ID
!pip -q install segmentation-models-pytorch rasterio geopandas
import os, glob, math, time, subprocess, numpy as np, pandas as pd, rasterio, torch, ee
import segmentation_models_pytorch as smp, matplotlib.pyplot as plt
from rasterio.warp import reproject, Resampling
from scipy.ndimage import binary_erosion
DEV = 'cuda' if torch.cuda.is_available() else 'cpu'; print('device:', DEV)

from google.colab import drive
drive.mount('/content/drive')
BASE = '/content/drive/MyDrive/Paper2_flood'
NEPAL = '/content/drive/MyDrive/Paper2_flood_nepal'
OUT = f'{BASE}/06_s2_validation'; os.makedirs(OUT, exist_ok=True)
STACK = f'{NEPAL}/koshi_terai_flood2024-10-01_pre2024-09-19_orbit12.tif'
DRY_S1 = sorted(glob.glob(f'{NEPAL}/koshi_terai_dry*_orbit12.tif'))
assert os.path.exists(STACK) and DRY_S1, 'Koshi stack or dry-season S1 file missing in Paper2_flood_nepal'

ee.Authenticate(); ee.Initialize(project=PROJECT)
AOI = ee.Geometry.Rectangle([86.85, 26.45, 87.15, 26.75])
S1_DATE = '2024-10-01'
print('ready')

## 1. How clear was Sentinel-2 after the flood?
`clear %` = share of the Koshi box not covered by cloud, cloud shadow or cirrus (scene classification band).

In [ ]:
s2 = ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED').filterBounds(AOI)

def clear_mask(im):
    scl = im.select('SCL')
    return scl.eq(2).Or(scl.gte(4).And(scl.lte(7)))

def day_mosaic(date):
    d = ee.Date(date); return s2.filterDate(d, d.advance(1, 'day')).mosaic()

days = sorted(set(s2.filterDate('2024-09-28', '2024-10-16').aggregate_array('system:time_start')
                  .map(lambda t: ee.Date(t).format('YYYY-MM-dd')).getInfo()))
rows = []
for d in days:
    im = day_mosaic(d)
    cov = im.select('SCL').gt(0).unmask(0).reduceRegion(ee.Reducer.mean(), AOI, 60).get('SCL')
    clr = clear_mask(im).unmask(0).reduceRegion(ee.Reducer.mean(), AOI, 60).get('SCL')
    rows.append({'date': d, 'coverage %': ee.Number(cov).multiply(100), 'clear %': ee.Number(clr).multiply(100)})
s2tab = pd.DataFrame([{k: (v.getInfo() if isinstance(v, ee.ComputedObject) else v) for k, v in r.items()} for r in rows]).round(0)
s2tab['days after S1'] = (pd.to_datetime(s2tab.date) - pd.Timestamp(S1_DATE)).dt.days
display(s2tab)

ok = s2tab[s2tab['clear %'] >= 50].copy()
if ok.empty:
    print('⚠️ No Sentinel-2 date is at least 50% clear - independent S2 labels are not possible; use hand labels instead.')
    S2_DATE = None
else:
    ok['rank'] = ok['clear %'] - 5 * ok['days after S1'].abs()   # prefer clear AND close to the S1 date
    S2_DATE = ok.sort_values('rank', ascending=False).iloc[0].date
    print('Using Sentinel-2 date:', S2_DATE)
S2_OVERRIDE = None   # e.g. '2024-10-04' to choose yourself
if S2_OVERRIDE: S2_DATE = S2_OVERRIDE

## 2. Build labels and export them
- `s2_water`: 1 = water (MNDWI > 0.1), 0 = dry (MNDWI < −0.1), 255 = cloud or ambiguous.
- `dry_water`: same, from the Feb–Apr 2024 cloud-free median (normal water).

In [ ]:
def mndwi(im): return im.normalizedDifference(['B3', 'B11'])

def label(index, valid):
    lab = ee.Image(255).where(index.gt(0.1), 1).where(index.lt(-0.1), 0)
    return lab.updateMask(valid).unmask(255)

flood_im = day_mosaic(S2_DATE)
s2_water = label(mndwi(flood_im), clear_mask(flood_im)).rename('s2_water')
dry_col = (s2.filterDate('2024-02-01', '2024-04-30').filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 30))
           .map(lambda im: mndwi(im).updateMask(clear_mask(im))))
dry_water = label(dry_col.median(), dry_col.count().gt(0)).rename('dry_water')
labels = ee.Image.cat([s2_water, dry_water]).toUint8().clip(AOI)

LAB = f'{NEPAL}/koshi_terai_s2labels_{S2_DATE}.tif'
if not os.path.exists(LAB):
    task = ee.batch.Export.image.toDrive(image=labels, description='koshi_s2_labels', folder='Paper2_flood_nepal',
            fileNamePrefix=f'koshi_terai_s2labels_{S2_DATE}', region=AOI, scale=10, crs='EPSG:32645', maxPixels=1e10)
    task.start()
    while task.status()['state'] in ('READY', 'RUNNING'):
        print(task.status()['state']); time.sleep(30)
    print('export:', task.status()['state'], task.status().get('error_message', ''))
    for _ in range(20):                      # wait for Drive to sync
        if os.path.exists(LAB): break
        time.sleep(15)
assert os.path.exists(LAB), 'label file not visible in Drive yet - wait a minute and rerun this cell'
print('labels ready:', LAB)

## 3. Load Koshi data on one grid

In [ ]:
BANDS = ['flood_VV', 'flood_VH', 'pre_VV', 'pre_VH', 'slope', 'hand', 'perm_water', 'ls_mask', 'quick_flood']
with rasterio.open(STACK) as s:
    b = dict(zip(BANDS, s.read().astype('float32'))); prof = s.profile
shape = b['flood_VV'].shape

def on_grid(path, n, dtype, fill):
    out = np.full((n,) + shape, fill, dtype)
    with rasterio.open(path) as src:
        for i in range(n):
            reproject(rasterio.band(src, i + 1), out[i], dst_transform=prof['transform'], dst_crs=prof['crs'],
                      resampling=Resampling.nearest, src_nodata=None)
    return out

dry_s1 = on_grid(DRY_S1[-1], 2, 'float32', 0)
lab = on_grid(LAB, 2, 'uint8', 255)
valid = np.ones(shape, bool)
for a in [b['flood_VV'], b['flood_VH'], dry_s1[0], dry_s1[1]]:
    valid &= np.isfinite(a) & (a != 0)
valid = binary_erosion(valid, iterations=3)
s2w, dryw = lab[0], lab[1]
print(f"S2 labelled pixels: {((s2w != 255) & valid).mean() * 100:.0f}% of the scene; "
      f"water share among them: {(s2w[valid & (s2w != 255)] == 1).mean() * 100:.1f}%")

## 4. Run models A and B on the flood and dry-season images

In [ ]:
ROOT = '/content/sen1floods11'; os.makedirs(ROOT, exist_ok=True)
if not os.path.exists(f'{ROOT}/S1Hand'):
    for p in ['data/flood_events/HandLabeled/S1Hand', 'splits/flood_handlabeled']:
        subprocess.run(['gsutil', '-m', '-q', 'cp', '-r', f'gs://sen1floods11/v1.1/{p}', f'{ROOT}/'], check=True)
CACHE = f'{BASE}/02_terrain/terrain_chips'

def prep(sar, slope, hand):
    sar = np.clip(np.nan_to_num(sar, nan=-50), -50, 1)
    t = np.stack([np.clip(np.nan_to_num(slope), 0, 60), np.log1p(np.clip(np.nan_to_num(hand), 0, 200))])
    return np.concatenate([sar, t]).astype('float32')

train = [f.replace('_S1Hand.tif', '') for f in pd.read_csv(f'{ROOT}/flood_handlabeled/flood_train_data.csv', header=None)[0]]
xs = []
for c in train:
    with rasterio.open(f'{ROOT}/S1Hand/{c}_S1Hand.tif') as s: sar = s.read().astype('float32')
    t = np.load(f'{CACHE}/{c}.npy'); xs.append(prep(sar, t[0], t[1]))
xs = np.stack(xs); MEAN, STD = xs.mean((0, 2, 3)), xs.std((0, 2, 3)) + 1e-6; del xs

def load_model(path, ch):
    m = smp.Unet('resnet34', encoder_weights=None, in_channels=ch, classes=2).to(DEV)
    m.load_state_dict(torch.load(path, map_location=DEV)); return m.eval()
MODELS = {'A_sar': (load_model(f'{BASE}/02_terrain/unet_A_sar.pt', 2), [0, 1]),
          'B_sar_terrain': (load_model(f'{BASE}/02_terrain/unet_B_sar_terrain.pt', 4), [0, 1, 2, 3])}

@torch.no_grad()
def predict(model, x, tile=512, stride=448, batch=8):
    C, H, W = x.shape
    Hp = max(tile, math.ceil((H - tile) / stride) * stride + tile); Wp = max(tile, math.ceil((W - tile) / stride) * stride + tile)
    xp = np.pad(x, ((0, 0), (0, Hp - H), (0, Wp - W)), mode='edge')
    prob = np.zeros((Hp, Wp), 'float32'); cnt = np.zeros((Hp, Wp), 'float32')
    pos = [(r, c) for r in range(0, Hp - tile + 1, stride) for c in range(0, Wp - tile + 1, stride)]
    for i in range(0, len(pos), batch):
        pb = pos[i:i + batch]
        p = torch.softmax(model(torch.tensor(np.stack([xp[:, r:r+tile, c:c+tile] for r, c in pb])).to(DEV)), 1)[:, 1].cpu().numpy()
        for (r, c), pi in zip(pb, p): prob[r:r+tile, c:c+tile] += pi; cnt[r:r+tile, c:c+tile] += 1
    return (prob / cnt)[:H, :W]

W = {}
for date, sar in [('flood', np.stack([b['flood_VV'], b['flood_VH']])), ('dry', dry_s1)]:
    x = (prep(sar, b['slope'], b['hand']) - MEAN[:, None, None]) / STD[:, None, None]; x[:, ~valid] = 0
    for m, (model, ch) in MODELS.items():
        W[(m, date)] = predict(model, x[ch]) > 0.5
print('predictions done')

## 5. Level 1: water extent on the flood date
Scored where Sentinel-2 gives a clear label. The S1 and S2 dates differ, so some disagreement is real change, not model error.

In [ ]:
def score(name, pred, truth, mask):
    p, t = pred[mask], truth[mask]
    tp, fp, fn = (p & t).sum(), (p & ~t).sum(), (~p & t).sum()
    pr, rc = tp / max(tp + fp, 1), tp / max(tp + fn, 1)
    return {'method': name, 'IoU': round(tp / max(tp + fp + fn, 1), 3), 'F1': round(2 * pr * rc / max(pr + rc, 1e-9), 3),
            'precision': round(pr, 3), 'recall': round(rc, 3), 'scored km²': round(mask.sum() / 1e4, 1)}

m1 = valid & (s2w != 255); truth1 = s2w == 1
level1 = pd.DataFrame([score('threshold (VV < −15 dB)', b['flood_VV'] < -15, truth1, m1),
                       score('A_sar', W[('A_sar', 'flood')], truth1, m1),
                       score('B_sar_terrain', W[('B_sar_terrain', 'flood')], truth1, m1)])
level1.to_csv(f'{OUT}/level1_water_extent.csv', index=False); level1

## 6. Level 2: new water (flood-date water that is not normal water)
Truth = S2 water on the flood date AND not water in the dry-season S2 median. Every method uses the **dry season** as its baseline.

In [ ]:
perm = b['perm_water'] > 0
m2 = valid & (s2w != 255) & (dryw != 255); truth2 = (s2w == 1) & (dryw == 0)
preds = {'change detection (vs dry)': (b['flood_VV'] < -15) & (b['flood_VV'] - dry_s1[0] < -3) & ~perm}
for m in MODELS: preds[m + ' (vs dry)'] = W[(m, 'flood')] & ~W[(m, 'dry')] & ~perm

z = glob.glob('/content/*FL20240928NPL*SHP*.zip') + glob.glob(f'{NEPAL}/*FL20240928NPL*SHP*.zip')
if z:
    import geopandas as gpd
    from rasterio.features import rasterize
    un = gpd.read_file(f'/vsizip/{z[0]}/FL20240928NPL_SHP/S1_20241001_FloodExtent_Koshi_Madhesh.shp').to_crs(prof['crs'])
    preds['UNOSAT'] = rasterize(((g, 1) for g in un.geometry), out_shape=shape, transform=prof['transform'],
                                fill=0, dtype='uint8').astype(bool)
else:
    print('UNOSAT zip not found in /content or Paper2_flood_nepal - skipped')

level2 = pd.DataFrame([score(k, v, truth2, m2) for k, v in preds.items()])
level2.insert(1, 'new water km² (scored area)', [round((v & m2).sum() / 1e4, 1) for v in preds.values()])
print(f'S2 new water in scored area: {(truth2 & m2).sum() / 1e4:.1f} km²')
level2.to_csv(f'{OUT}/level2_new_water.csv', index=False); level2

## 7. Maps

In [ ]:
def rgb(pred, truth, mask):
    im = np.full(shape + (3,), 0.85); im[mask] = 1
    im[mask & pred & truth] = [0.1, 0.6, 0.1]; im[mask & pred & ~truth] = [0.85, 0.1, 0.1]; im[mask & ~pred & truth] = [0.1, 0.3, 0.9]
    return im[::3, ::3]

keys = list(preds)
fig, ax = plt.subplots(1, len(keys), figsize=(5 * len(keys), 6))
for a, k in zip(np.atleast_1d(ax), keys):
    a.imshow(rgb(preds[k], truth2, m2)); a.set_title(k, fontsize=10); a.axis('off')
fig.suptitle(f'New water vs Sentinel-2 {S2_DATE}: green correct, red false alarm, blue missed, grey = no clear label')
plt.tight_layout(); plt.savefig(f'{OUT}/level2_maps.png', dpi=110)